# Dataset Preparation — Classification
Collects JPG images from two source folders (two classes), shuffles each class independently, converts to PNG, and splits into **train / test / validation** at a **6 : 2 : 2** ratio.

**Output structure:**
```
output_base/
├── train/
│   ├── class1/
│   └── class2/
├── test/
│   ├── class1/
│   └── class2/
└── validation/
    ├── class1/
    └── class2/
```
Compatible with `torchvision.datasets.ImageFolder`, `tf.keras.utils.image_dataset_from_directory`, etc.

## 1 · Install dependency

In [2]:
!pip install Pillow -q

## 2 · Imports

In [29]:
import random
from pathlib import Path
from PIL import Image

## 3 · Configuration — edit these

In [36]:
# Source folders — each folder = one class
SOURCES = {
    "AnnualCrop_Negative": r"F:\image_processing\project\filtered\Negative\AnnualCrop_Neg",   # <-- change this
    "PermanentCrop_Negative": r"F:\image_processing\project\filtered\Negative\PermanentCrop_Neg",   # <-- change this
}

OUTPUT_BASE = r"F:\image_processing\dataset\Negative"     # <-- change this

SPLIT_RATIO = (6, 2, 2)             # train : test : validation
RANDOM_SEED = 42

## 4 · Collect JPG images per class

In [37]:
exts = {".png"}
class_images = {}   # { class_name: [Path, ...] }

for class_name, folder in SOURCES.items():
    p = Path(folder)
    if not p.is_dir():
        raise NotADirectoryError(f"Not a directory: {folder}")
    found = [f for f in p.iterdir() if f.suffix.lower() in exts]
    class_images[class_name] = found
    print(f"{class_name} ({folder}): {len(found)} PNG file(s) found")

print(f"\nTotal images: {sum(len(v) for v in class_images.values())}")

AnnualCrop_Negative (F:\image_processing\project\filtered\Negative\AnnualCrop_Neg): 2500 PNG file(s) found
PermanentCrop_Negative (F:\image_processing\project\filtered\Negative\PermanentCrop_Neg): 2500 PNG file(s) found

Total images: 5000


## 5 · Shuffle each class independently

In [38]:
random.seed(RANDOM_SEED)
for class_name, images in class_images.items():
    random.shuffle(images)
    print(f"{class_name}: shuffled {len(images)} images")

AnnualCrop_Negative: shuffled 2500 images
PermanentCrop_Negative: shuffled 2500 images


## 6 · Split each class 6 : 2 : 2

In [39]:
def split_list(items, ratios):
    """Split a list into (train, test, val) according to ratios."""
    total  = sum(ratios)
    n      = len(items)
    n_tr   = round(n * ratios[0] / total)
    n_te   = round(n * ratios[1] / total)
    return items[:n_tr], items[n_tr:n_tr+n_te], items[n_tr+n_te:]

splits = {}   # { class_name: {"train": [...], "test": [...], "validation": [...]} }

print(f"{'Class':<10} {'Train':>7} {'Test':>7} {'Val':>7} {'Total':>7}")
print("-" * 40)
for class_name, images in class_images.items():
    tr, te, va = split_list(images, SPLIT_RATIO)
    splits[class_name] = {"train": tr, "test": te, "validation": va}
    print(f"{class_name:<10} {len(tr):>7} {len(te):>7} {len(va):>7} {len(images):>7}")

Class        Train    Test     Val   Total
----------------------------------------
AnnualCrop_Negative    1500     500     500    2500
PermanentCrop_Negative    1500     500     500    2500


## 7 · Convert JPG → PNG and save into class subfolders

In [40]:
import shutil

base       = Path(OUTPUT_BASE)
total_ok   = 0
total_fail = 0

for class_name, split_dict in splits.items():
    index = 0   # per-class sequential index
    for split_name, paths in split_dict.items():
        dest_dir = base / split_name / class_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        ok = fail = 0

        for src in paths:
            dest = dest_dir / f"{index:05d}.png"
            try:
                shutil.copy2(src, dest)
                ok += 1
            except Exception as e:
                print(f"  [WARN] {src.name}: {e}")
                fail += 1
            index += 1

        print(f"[{split_name}/{class_name}]  saved: {ok}  |  failed: {fail}  ->  {dest_dir}")
        total_ok   += ok
        total_fail += fail

print(f"\nDone — copied: {total_ok}  |  failed: {total_fail}")
print(f"Output base: {base.resolve()}")

[train/AnnualCrop_Negative]  saved: 1500  |  failed: 0  ->  F:\image_processing\dataset\Negative\train\AnnualCrop_Negative
[test/AnnualCrop_Negative]  saved: 500  |  failed: 0  ->  F:\image_processing\dataset\Negative\test\AnnualCrop_Negative
[validation/AnnualCrop_Negative]  saved: 500  |  failed: 0  ->  F:\image_processing\dataset\Negative\validation\AnnualCrop_Negative
[train/PermanentCrop_Negative]  saved: 1500  |  failed: 0  ->  F:\image_processing\dataset\Negative\train\PermanentCrop_Negative
[test/PermanentCrop_Negative]  saved: 500  |  failed: 0  ->  F:\image_processing\dataset\Negative\test\PermanentCrop_Negative
[validation/PermanentCrop_Negative]  saved: 500  |  failed: 0  ->  F:\image_processing\dataset\Negative\validation\PermanentCrop_Negative

Done — copied: 5000  |  failed: 0
Output base: F:\image_processing\dataset\Negative


## 8 · Sanity check

In [41]:
print(f"{'Split':<12} {'Class':<10} {'PNG count':>10}")
print("-" * 35)
for split_name in ["train", "test", "validation"]:
    for class_name in SOURCES:
        folder = base / split_name / class_name
        count  = len(list(folder.glob("*.png")))
        print(f"{split_name:<12} {class_name:<10} {count:>10}")

Split        Class       PNG count
-----------------------------------
train        AnnualCrop_Negative       1500
train        PermanentCrop_Negative       1500
test         AnnualCrop_Negative        500
test         PermanentCrop_Negative        500
validation   AnnualCrop_Negative        500
validation   PermanentCrop_Negative        500


## 9 · Load with PyTorch ImageFolder (example)

In [ ]:
# Uncomment and run once you have torchvision installed

# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
#
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406],
#                          [0.229, 0.224, 0.225]),
# ])
#
# train_ds = datasets.ImageFolder(str(base / 'train'),      transform=transform)
# test_ds  = datasets.ImageFolder(str(base / 'test'),       transform=transform)
# val_ds   = datasets.ImageFolder(str(base / 'validation'), transform=transform)
#
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
# test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)
# val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
#
# print("Classes:", train_ds.classes)
# print(f"train: {len(train_ds)}  test: {len(test_ds)}  val: {len(val_ds)}")